# Tiền xử lý

In [2]:
import os
import shutil
import hashlib
from tqdm import tqdm  

# Định nghĩa paths nguồn
dataset1_root = '/kaggle/input/firesmoke-detection-yolo-v9/Fire and Smoke Dataset'
dataset2_root = '/kaggle/input/smoke-fire-detection-yolo'

# Paths cho các split
splits = ['train', 'val', 'test'] 
dataset1_splits = {'train': 'train', 'val': 'valid', 'test': 'test'}


output_root = '/kaggle/working/DatasetAll'
os.makedirs(output_root, exist_ok=True)

# Dict để lưu hash hình ảnh copied
image_hashes = {}

# Hàm tính hash của file
def compute_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

# Hàm copy files từ một dataset, xử lý trùng lặp và nhãn
def copy_dataset(dataset_root, splits_map, is_dataset2=False):
    for split in splits:
        src_split = splits_map.get(split, split)  # Cho dataset1, val -> valid
        src_images = os.path.join(dataset_root, src_split, 'images')
        src_labels = os.path.join(dataset_root, src_split, 'labels')
        
        if not os.path.exists(src_images):
            print(f"Không tìm thấy {src_images}, skip.")
            continue
        
        dest_images = os.path.join(output_root, split, 'images')
        dest_labels = os.path.join(output_root, split, 'labels')
        os.makedirs(dest_images, exist_ok=True)
        os.makedirs(dest_labels, exist_ok=True)
        
        image_files = [f for f in os.listdir(src_images) if f.endswith(('.jpg', '.png', '.jpeg'))]
        for img_file in tqdm(image_files, desc=f"Copy {split} from dataset"):
            src_img = os.path.join(src_images, img_file)
            img_hash = compute_hash(src_img)
            
            if img_hash in image_hashes:
                print(f"Trùng lặp: {img_file} giống với {image_hashes[img_hash]}, skip.")
                continue
            
            # Tách phần mở rộng ngay từ đầu
            base, ext = os.path.splitext(img_file)
            
            # Copy image
            dest_img = os.path.join(dest_images, img_file)
            if os.path.exists(dest_img):  # Rename nếu tên trùng nhưng nội dung khác
                dest_img = os.path.join(dest_images, f"{base}_dup{ext}")
            shutil.copy(src_img, dest_img)
            image_hashes[img_hash] = img_file
            
            # Copy và xử lý label
            label_file = base + '.txt'  # Sử dụng base thay vì os.path.splitext lại
            src_label = os.path.join(src_labels, label_file)
            if os.path.exists(src_label):
                dest_label = os.path.join(dest_labels, os.path.basename(dest_img).replace(ext, '.txt'))
                with open(src_label, 'r') as f:
                    lines = f.readlines()
                if is_dataset2:  # Hoán đổi class_id cho dataset2
                    new_lines = []
                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            class_id = int(parts[0])
                            new_class_id = 1 - class_id  # 0 (smoke) -> 1, 1 (fire) -> 0
                            parts[0] = str(new_class_id)
                            new_lines.append(' '.join(parts) + '\n')
                    lines = new_lines
                with open(dest_label, 'w') as f:
                    f.writelines(lines)

# Tét trùng
copy_dataset(dataset1_root, dataset1_splits, is_dataset2=False)
copy_dataset(dataset2_root + '/data', {'train': 'train', 'val': 'val', 'test': 'test'}, is_dataset2=True)

# Tạo data.yaml mới
yaml_content = """
train: /kaggle/working/DatasetAll/train/images
val: /kaggle/working/DatasetAll/val/images
test: /kaggle/working/DatasetAll/test/images

nc: 2
names: ['fire', 'smoke']
"""
with open(os.path.join(output_root, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

# Đếm files để kiểm tra
for split in splits:
    img_count = len(os.listdir(os.path.join(output_root, split, 'images')))
    label_count = len(os.listdir(os.path.join(output_root, split, 'labels')))
    print(f"{split}: {img_count} images, {label_count} labels")

print("Gộp hoàn tất! Dataset mới tại /kaggle/working/DatasetAll")

Copy train from dataset:   2%|▏         | 576/35695 [00:07<07:40, 76.33it/s]

Trùng lặp: 962_jpg.rf.66aacee865fb3310afb0470793655970.jpg giống với 962_jpg.rf.a660e5abae9062b00bcd9dd7d5ab47e6.jpg, skip.


Copy train from dataset:   2%|▏         | 718/35695 [00:08<07:50, 74.42it/s]

Trùng lặp: 91-Copy_jpg.rf.bbea1ee66275241c30775656844da625.jpg giống với 91-Copy_jpg.rf.1c62a56d046dcb86887d48fd918a6c1a.jpg, skip.


Copy train from dataset:   2%|▏         | 737/35695 [00:09<07:50, 74.34it/s]

...

Trùng lặp: fire1-597-_jpg.rf.01750a41bc2889cd60c2a99aeabdfd04.jpg giống với fire1-597-_jpg.rf.c3e0600707cc20902da79c229d209b11.jpg, skip.
Trùng lặp: IMG_5398-1-_jpeg_jpg.rf.9e11e4b91ba43cfb6baecb041978db4c.jpg giống với IMG_5398-1-_jpeg_jpg.rf.9fcf1139d20fcb2e86b30941deae9cda.jpg, skip.


Copy test from dataset: 100%|██████████| 2255/2255 [00:27<00:00, 83.00it/s]

Trùng lặp: Img_1974_jpg.rf.3cb333424595ba991d3c7d2db676fd53.jpg giống với Img_1974_jpg.rf.38c0e5fad1c752b29238a02b794a07b5.jpg, skip.
Trùng lặp: Img_1164_jpg.rf.321d8d0ee892624c7ee9f88e2bf39f01.jpg giống với Img_1164_jpg.rf.ca43ac1efed3056b7579237949ae3d7b.jpg, skip.
Trùng lặp: new_fire_fire-55_png_jpg.rf.3f90821a2bd6a31d8dff64bb2d80b7cf.jpg giống với new_fire_fire-55_png_jpg.rf.e52f65c04f8ec61bff6e81898395fd4a.jpg, skip.
Trùng lặp: Img_1980_jpg.rf.68e499d3109eb2ace7b36280dc111049.jpg giống với Img_1980_jpg.rf.462c7b20a00524f117dfd360e90b80d4.jpg, skip.
Trùng lặp: fire1-939-_jpg.rf.ef213317a7923bbb4fb4a3af9113b62b.jpg giống với fire1-939-_jpg.rf.a877ace52c093358884814a373758122.jpg, skip.



Copy test from dataset: 100%|██████████| 4306/4306 [01:31<00:00, 47.08it/s]


train: 47790 images, 47789 labels
val: 7393 images, 7393 labels
test: 6244 images, 6244 labels
Gộp hoàn tất! Dataset mới tại /kaggle/working/DatasetAll


In [4]:
%%writefile /kaggle/working/data.yaml
train: /kaggle/working/DatasetAll/train/images
val: /kaggle/working/DatasetAll/val/images
test: /kaggle/working/DatasetAll/test/images

nc: 2
names: ['fire', 'smoke']

Overwriting /kaggle/working/data.yaml


# Training

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninst

In [2]:
import os
import yaml
from ultralytics import YOLO
import torch
from torch.cuda.amp import autocast, GradScaler

# ============================
# Cấu hình thiết bị
# ============================
device = [0,1]

# ============================
# Đọc file cấu hình dataset
# ============================
data_yaml_path = "/kaggle/working/data.yaml"
with open(data_yaml_path, 'r') as file:
    data_config = yaml.safe_load(file)

print(f"Số lớp: {data_config['nc']}")
print(f"Tên lớp: {data_config['names']}")

# ============================
# Khởi tạo mô hình
# ============================
model = YOLO("yolo11n.pt") 
checkpoint_dir = '/kaggle/working/runs/fire_smoke_detection_v1/weights'
resume_training = False

if os.path.exists(checkpoint_dir):
    if 'last.pt' in os.listdir(checkpoint_dir):
        print("Tìm thấy checkpoint, sẽ tiếp tục training...")
        resume_training = True
        model = YOLO(os.path.join(checkpoint_dir, 'last.pt'))

# ============================
# Cấu hình training
# ============================
training_config = {
    'data': data_yaml_path,
    'epochs': 160,
    'imgsz': 640,
    'batch': 32,
    'workers': 8,
    'device': device,
    'save': True,
    'save_period': 5,
    'project': '/kaggle/working/runs',
    'name': 'fire_smoke_detection_v1',
    'exist_ok': True,
    'patience': 25,
    'lr0': 0.005,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 10,
    'amp': True,
    'cos_lr': True,
    'nms': True,
    'augment': True,
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.5,
    'shear': 0.0,
    'mosaic': 0.8,
    'mixup': 0.1,
    'hsv_h': 0.025,
    'hsv_s': 0.3,
    'hsv_v': 0.4,
    'fliplr': 0.5,
    'flipud': 0.0,
    'copy_paste': 0.1,
    'perspective': 0.0001
}

# ============================
# Huấn luyện mô hình với mixed precision
# ============================
try:
    scaler = GradScaler(enabled=training_config['amp'])
    if resume_training:
        print("==== Tiếp tục huấn luyện từ checkpoint ====")
        results = model.train(resume=True, **training_config)
    else:
        print("==== Bắt đầu huấn luyện mới ====")
        with autocast(enabled=training_config['amp']):
            results = model.train(**training_config)

except KeyboardInterrupt:
    print("Huấn luyện bị ngắt. Checkpoint đã lưu trong runs/fire_smoke_detection_v1/weights/")
except Exception as e:
    print(f"Lỗi trong quá trình huấn luyện: {e}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Số lớp: 2
Tên lớp: ['fire', 'smoke']
Tìm thấy checkpoint, sẽ tiếp tục training...


/tmp/ipykernel_36/3538529492.py:79: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=training_config['amp'])


==== Tiếp tục huấn luyện từ checkpoint ====
Ultralytics 8.3.199 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=160, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.025, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/kaggle/working/runs/fire_smoke_detection_v1/weights/last.pt, momentum=0.937, 

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       7393       9037      0.785      0.672      0.763      0.465
                  fire       3533       5929      0.778      0.662      0.757      0.427
                 smoke       2597       3108      0.791      0.682      0.769      0.502
Speed: 0.2ms preprocess, 6.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/fire_smoke_detection_v1


# Export

In [ ]:
import os
import yaml
from ultralytics import YOLO
import torch

# ============================
# Cấu hình
# ============================
data_yaml_path = "/kaggle/working/data.yaml"
weights_path = "/kaggle/working/runs/fire_smoke_detection_v   /weights/last.pt"
output_dir = "/kaggle/working/exported_models"
os.makedirs(output_dir, exist_ok=True)

# Đọc cấu hình
with open(data_yaml_path, 'r') as file:
    data_config = yaml.safe_load(file)
print(f"Số lớp: {data_config['nc']}, Tên lớp: {data_config['names']}")

# ============================
# Khởi tạo mô hình
# ============================
model = YOLO(weights_path)

# ============================
# Đánh giá mô hình
# ============================
print("==== Bắt đầu đánh giá mô hình trên tập test ====")

results = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=24,
    device=[0, 1],
    save_json=True,
)


print(f"Precision: {results.box.map:.4f}")
print(f"Recall: {results.box.map75:.4f}")
print(f"mAP@0.5: {results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {results.box.map:.4f}")
print(f"Kết quả chi tiết đã được lưu tại: {results.save_dir}")

# ============================
# Xuất mô hình
# ============================
print("==== Bắt đầu xuất mô hình ====")
export_formats = ['pt', 'onnx', 'tflite']

for fmt in export_formats:
    try:
        export_path = os.path.join(output_dir, f"fire_smoke_detection_v1.{fmt}")
        model.export(
            format=fmt,
            imgsz=640,
            batch=1,
            dynamic=True,
            device=[0]
        )
        print(f"Đã xuất mô hình sang {export_path}")
    except Exception as e:
        print(f"Lỗi khi xuất sang {fmt}: {e}")

print("Hoàn tất đánh giá và xuất mô hình! Kết quả và mô hình tại:", output_dir)

In [ ]:
import os
os.remove('/kaggle/working/file.zip')

!zip -r file.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/yolo11n.pt (deflated 11%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/DatasetAll/ (stored 0%)
  adding: kaggle/working/DatasetAll/val/ (stored 0%)
  adding: kaggle/working/DatasetAll/val/images/ (stored 0%)
  adding: kaggle/working/DatasetAll/val/images/WEB01902.jpg (deflated 0%)
  adding: kaggle/working/DatasetAll/val/images/WEB06648.jpg (deflated 1%)
  adding: kaggle/working/DatasetAll/val/images/FireDataThree17518_png.rf.ac6fa5a9efaf4c74b67b97aa80f50d66.jpg (deflated 2%)
  adding: kaggle/working/DatasetAll/val/images/WhatsApp-Video-2024-01-12-at-18_01_23_mp4-211_jpg.rf.eff3c259b2bd2bf49044ed386f61c2f6.jpg (deflated 4%)
  adding: kaggle/working/DatasetAll/val/images/IMG_5788_jpeg_jpg.rf.2874d2ba4538617f7c309a107cddea6d.jpg (deflated 2%)
  adding: kaggle/working/DatasetAll/val/images/WEB02546.jpg (deflated 7%)
  adding: kaggle/working/DatasetAll/val/images/Chemical-explosion-sparks-fac